# R3maJ Kaggle Notebook v4

Kaggle T4×2 training setup for R3maJ with Google Drive replay/checkpoint restore and **256 games**.

The notebook verifies both T4 GPUs, but it does **not** fake a collector/learner GPU split: the current R3maJ executable exposes only the verified single `--device cuda` path. A true `GPU 0 → collection / GPU 1 → learner` architecture requires code-level support in R3maJ/GigaLearn and must be implemented/tested before enabling it.

In [ ]:
# 1. Clone R3maJ
import os, subprocess
ROOT='/kaggle/working/R3maJ'
REPO='https://github.com/vfxjamer/R3maJ.git'
if not os.path.isdir(os.path.join(ROOT,'.git')):
    subprocess.run(['git','clone','--depth','1',REPO,ROOT],check=True)
else:
    subprocess.run(['git','-C',ROOT,'pull'],check=False)
print(ROOT)


In [ ]:
# 2. Download the complete R3maJ Drive folder (replay + checkpoints)
!pip install -q gdown
import gdown, os, shutil
DRIVE_FOLDER_ID='1ktqAHT6wwYCyyRA4REBN2kZyFqeCbvKh'
DOWNLOAD_DIR='/kaggle/working/R3maJ_drive'
LOCAL_ROOT='/kaggle/working/R3maJ/build'
LOCAL_REPLAY=f'{LOCAL_ROOT}/serialized_replays.bin'
LOCAL_CHECKPOINTS=f'{LOCAL_ROOT}/checkpoints'
os.makedirs(LOCAL_ROOT,exist_ok=True)
if not os.path.exists(DOWNLOAD_DIR):
    gdown.download_folder(f'https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}',output=DOWNLOAD_DIR,quiet=False,use_cookies=False)
print('Drive folder downloaded to:',DOWNLOAD_DIR)
for root,dirs,files in os.walk(DOWNLOAD_DIR):
    level=root.replace(DOWNLOAD_DIR,'').count(os.sep)
    indent='  '*level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files[:20]: print(f'{indent}  {f}')


In [ ]:
# 3. Restore replay + checkpoints into the locations R3maJ expects
import os, shutil
def find_file(base,name):
    for root,dirs,files in os.walk(base):
        if name in files: return os.path.join(root,name)
    return None
def find_dir(base,name):
    for root,dirs,files in os.walk(base):
        if name in dirs: return os.path.join(root,name)
    return None
src_replay=find_file(DOWNLOAD_DIR,'serialized_replays.bin')
src_checkpoints=find_dir(DOWNLOAD_DIR,'checkpoints')
assert src_replay,'serialized_replays.bin not found in shared R3maJ folder.'
assert src_checkpoints,'checkpoints folder not found in shared R3maJ folder.'
shutil.copy2(src_replay,LOCAL_REPLAY)
if os.path.exists(LOCAL_CHECKPOINTS): shutil.rmtree(LOCAL_CHECKPOINTS)
shutil.copytree(src_checkpoints,LOCAL_CHECKPOINTS)
print('Replay:',LOCAL_REPLAY)
print('Replay GB:',round(os.path.getsize(LOCAL_REPLAY)/(1024**3),3))
print('Checkpoint entries:',sorted(os.listdir(LOCAL_CHECKPOINTS))[:20])


### Drive layout expected
`R3maJ/serialized_replays.bin`
`R3maJ/checkpoints/<checkpoint directories>`

The single folder ID above is used to restore both.

In [ ]:
# 4. Install build dependencies + inspect both GPUs
import subprocess,os
subprocess.run(['apt-get','update','-qq'],check=False)
subprocess.run(['apt-get','install','-y','-qq','build-essential','cmake','git','libpython3-dev','pkg-config'],check=False)
import torch
print('torch:',torch.__version__)
print('CUDA:',torch.cuda.is_available(),torch.version.cuda)
print('GPU count:',torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}')
assert torch.cuda.device_count() >= 2,'Expected Kaggle T4×2 runtime.'
print('Both T4s are visible. R3maJ itself remains on the verified single-device path until explicit multi-GPU support is added.')
subprocess.run(['nvidia-smi'],check=False)


In [ ]:
# 5. Configure + build
import os,subprocess,torch
os.chdir(ROOT)
prefix=os.path.dirname(torch.__file__)
subprocess.run(['cmake','-S','.','-B','build','-DCMAKE_BUILD_TYPE=Release',f'-DTORCH_INSTALL_PREFIX={prefix}'],check=True)
subprocess.run(['cmake','--build','build','-j',str(os.cpu_count() or 2)],check=True)
EXE=os.path.join(ROOT,'build','R3maJ')
print('Binary:',EXE,os.path.exists(EXE))


In [ ]:
# 6. Verify restored data
assert os.path.exists(EXE),'R3maJ binary missing.'
assert os.path.exists(LOCAL_REPLAY),'Replay missing.'
assert os.path.isdir(LOCAL_CHECKPOINTS),'Checkpoint directory missing.'
print('READY')
print('Games:',256)
print('Checkpoint entries:',len([x for x in os.listdir(LOCAL_CHECKPOINTS) if not x.startswith('.')]))


In [ ]:
# 7. Start training — 256 games
import os,subprocess
os.chdir(os.path.join(ROOT,'build'))
TRAIN_ARGS=['--device','cuda','--save-dir','checkpoints','--games','256','--replays','serialized_replays.bin']
print('Launching:','./R3maJ',*TRAIN_ARGS)
proc=subprocess.Popen(['./R3maJ']+TRAIN_ARGS)
print('Training PID:',proc.pid)


## GPU utilization / T4×2 note

Kaggle provides two Tesla T4 GPUs, but the current R3maJ executable has no verified multi-GPU collector/learner option. Therefore v4 does **not** launch two independent training processes or invent a `cuda:0,cuda:1` argument.

The intended future architecture is:
`T4 #0 → game collection/inference → experience queue → T4 #1 → PPO learner`

That requires explicit runtime support and synchronization in R3maJ/GigaLearn. Until that is implemented, the safe configuration is one R3maJ process with `--device cuda` and **256 games**.